# USER PARAMETERS

**Question.** How does a function change along the Mellin/Fourier reduction? Set the half-line weight, the log grid, and the multiplier mode in the next cell. The five visible stages are \(f(x)\), \(g(t)\), \(\widehat g(\lambda)\), \(a(\lambda)\widehat g(\lambda)\), and the reconstructed output. Run all cells from a clean Python 3.12.2 kernel.

In [1]:
# USER PARAMETERS
p = 2.0                      # exponent of the weighted Lp space
delta = 0.0                  # weight x^delta; require -1/p < delta < 1-1/p
T_MIN = -7.0                 # left log-coordinate endpoint
T_MAX = 7.0                  # right log-coordinate endpoint
N_GRID = 1024                # FFT grid points; at least 256
SIGMA = 1.1                  # width of the Gaussian g(t)
SYMBOL_MODE = "smooth_demo"  # "smooth_demo" or "paper_coth"
FPS = 24                     # output video frame rate
SPEED_FACTOR = 1.0           # values >1 shorten each stage
STAGE_DURATION = 1.5 / SPEED_FACTOR
HOLD_END = 1.2 / SPEED_FACTOR
OUTPUT_NAME = "mellin_fourier_pipeline"
OUTPUT_GIF = OUTPUT_NAME + ".gif"
OUTPUT_MP4 = OUTPUT_NAME + ".mp4"
OUTPUT_PNG = OUTPUT_NAME + ".png"
OUTPUT_PDF = OUTPUT_NAME + ".pdf"


## Exact coordinate and Fourier conventions

Let \(\nu=1/p+\delta\), \(x=e^t\), and \(E_\nu f(t)=e^{\nu t}f(e^t)\). We choose \(g(t)=\exp[-t^2/(2\sigma^2)]\) and define \(f(x)=x^{-\nu}g(\log x)\), so this coordinate change is visible at the first two stages. The admissible weight interval implies \(0<\nu<1\).

Our Fourier convention is
\[
\widehat g(\lambda)=\int_{\mathbb R}g(t)e^{-i\lambda t}\,dt,\qquad
g(t)=\frac1{2\pi}\int_{\mathbb R}\widehat g(\lambda)e^{i\lambda t}\,d\lambda.
\]
The output is \(E_\nu^{-1}\mathcal F^{-1}(a\widehat g)\). For the default numerical demonstration, \(a(\lambda)=0.20+0.80\exp[-(\lambda/0.70)^2/2]\), a smooth low-pass multiplier chosen for visual clarity. The article's exact diagonal Mellin multiplier in (3.25) is \(a_{\rm K25}(\lambda)=\coth[\pi(\lambda+i\nu)]\). Switching SYMBOL_MODE to paper_coth uses that multiplier numerically; the code reports its maximum modulus and raises if nonfinite values appear. No numerical mode is silently substituted for another.

In [2]:
from pathlib import Path
from dataclasses import replace
import sys
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
elif not (project_dir / "src" / "render_mellin_fourier.py").exists():
    project_dir = project_dir / "karlovich_2025_cusps"
if not (project_dir / "src" / "render_mellin_fourier.py").exists():
    raise FileNotFoundError("Open the notebook from its project or repository folder.")
sys.path.insert(0, str(project_dir / "src"))

from render_mellin_fourier import (
    PipelineConfig, compute_pipeline, render_assets, render_frame
)
config = PipelineConfig(
    p=p, delta=delta, t_min=T_MIN, t_max=T_MAX,
    n_grid=N_GRID, sigma=SIGMA, symbol_mode=SYMBOL_MODE,
    fps=FPS, stage_duration=STAGE_DURATION, hold_end=HOLD_END,
)
data = compute_pipeline(config)
paper_probe = compute_pipeline(replace(config, symbol_mode="paper_coth"))
print("Python:", sys.executable, sys.version.split()[0])
print("nu =", config.nu, "; mode =", SYMBOL_MODE)
print("Relative Lp-isometry quadrature error:", data.norm_relative_error)
print("Relative FFT inverse error:", data.inverse_relative_error)
print("Max |paper_coth| on grid:", np.max(np.abs(paper_probe.multiplier)))
assert data.norm_relative_error < 1e-3
assert data.inverse_relative_error < 1e-11
assert np.all(np.isfinite(paper_probe.multiplier))


Python: /home/enriquedo/.pyenv/versions/3.12.2/bin/python 3.12.2
nu = 0.5 ; mode = smooth_demo
Relative Lp-isometry quadrature error: 3.115365216356464e-05
Relative FFT inverse error: 3.3868974854406617e-16
Max |paper_coth| on grid: 1.0000000000000002


## Observe the whole chain

The plots change variables and objects at each step. The first and fifth panels use logarithmic horizontal axes; the middle plots use the additive coordinates \(t\) and \(\lambda\). The symbol is multiplied into the Fourier transform before inversion. This is a numerical illustration of the representation, not a proof of a Fredholm theorem.

In [3]:
preview = render_frame(config, data, 5)
display(preview)
plt.close(preview)


<Figure size 1280x720 with 5 Axes>

## Export stages

The animation reveals one stage at a time and leaves the five-panel chain visible at the end. PNG and vector PDF contain that final frame. The [canonical renderer](../src/render_mellin_fourier.py) contains the transform conventions, numerical checks, and export code.

In [4]:
assets = render_assets(config, project_dir / "outputs", OUTPUT_NAME)
assert assets["gif"].name == OUTPUT_GIF
assert assets["mp4"].name == OUTPUT_MP4
assert assets["png"].name == OUTPUT_PNG
assert assets["pdf"].name == OUTPUT_PDF
for kind in ("gif", "mp4", "png", "pdf"):
    path = assets[kind]
    print(kind, path.relative_to(project_dir), path.stat().st_size, "bytes")
print("Frames:", assets["frames"], "Duration:", assets["duration_seconds"], "s")
print("Norm check:", assets["norm_relative_error"],
      "FFT inversion:", assets["inverse_relative_error"])


gif outputs/mellin_fourier_pipeline.gif 147989 bytes
mp4 outputs/mellin_fourier_pipeline.mp4 53185 bytes
png outputs/mellin_fourier_pipeline.png 153386 bytes
pdf outputs/mellin_fourier_pipeline.pdf 29219 bytes
Frames: 209 Duration: 8.708333333333334 s
Norm check: 3.115365216356464e-05 FFT inversion: 3.3868974854406617e-16


## Interpretation and sources

The smooth_demo mode demonstrates a complete numerical pipeline with a deliberately chosen smooth symbol. The paper_coth mode substitutes the actual multiplier of Karlovich (2025), (3.25); its output may be complex. The exact theoretical relation is \(M^0(a)=E_\nu^{-1}\mathcal F^{-1}a\mathcal F E_\nu\), using the conventions above. The finite FFT and sampled figures do not establish boundedness, compactness, or Fredholmness.

**Primary source:** Karlovich (2025), (3.3)–(3.4), (3.24)–(3.28). **Classical context checked in the local literature index:** Duduchava, *Convolutions*, 1987, Definition (3.1), Theorems 3.1–3.2. Those theorems concern their stated algebra; this notebook makes no claim that they prove the 2025 theorem.

<!-- resultado-verificado: mellin_fourier -->